In [ ]:
# ===========================================================================
# UA-SPEECH MODEL TRAINING - interactive driver
#
# All training logic lives in src/training/; this notebook only calls it and
# stores results, matching notebooks/01_data_pipeline.ipynb's convention -
# functions and architecture belong in src/, only the act of running training
# and storing models happens here.
#
#   src/training/models.py     model factory: acoustic / deep_frozen / deep_lora / fusion
#   src/training/runner.py     TrainingConfig, run_training() - the fold loop
#   src/training/baseline.py   Phase 2: frozen wav2vec + linear SVM baseline
#   src/training/engine.py     one epoch: AMP, gradient clipping, optimizer
#   src/training/metrics.py    accuracy / precision / recall / specificity / F1 / AUROC
#   src/training/reporting.py  predictions / metrics / confusion-matrix / ROC / embeddings I/O
#
# See ROADMAP.md for the phase plan this notebook implements (Phase 1 sanity
# check, Phase 2 baseline reproduction and comparison).
# ===========================================================================

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.console import print_header, print_kv
from src.training.data import load_manifest

config.ensure_directories()

df_m6 = load_manifest()

print_header("UA-Speech Training Notebook")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))
print_kv("Speakers", df_m6["Speaker_ID"].nunique())

In [ ]:
# STAGE 1 - Pipeline sanity check ("smoke test"). Trains the cheapest model
# (MFCC-only) for one fold, one epoch, on a tiny slice of data. This is NOT a
# real result - it exists to confirm the whole chain (model init, optimizer,
# scheduler, AMP, gradient clipping, early stopping, checkpointing,
# TensorBoard logging, prediction/metric/confusion-matrix/ROC/embedding
# writers) actually runs end to end before spending GPU time on a real run.
from src.training.runner import TrainingConfig, run_training

smoke_cfg = TrainingConfig(
    task="detection", model="acoustic",
    epochs=1, max_folds=1, limit_samples=24,
    run_name="_smoke_test",
)
run_training(df_m6, smoke_cfg)

In [ ]:
# STAGE 2 - Phase 2, step 1: reproduce the ICASSP base paper's feature
# extractor. Frozen wav2vec 2.0 (no LoRA, no fine-tuning) -> 768-dim
# embedding per utterance. The embedding is identical across every LOSO
# fold, so it is extracted once here and cached to outputs/embeddings/ -
# only the SVM is refit per fold in Stage 3.
from src.training.baseline import extract_frozen_embeddings

frozen_embeddings = extract_frozen_embeddings(df_m6, batch_size=16)
print(f"Frozen embeddings: {frozen_embeddings.shape}")

In [ ]:
# STAGE 3 - Phase 2, step 2: frozen wav2vec 2.0 -> linear SVM, evaluated
# across the full 28-fold LOSO detection protocol - exactly the base paper's
# pipeline. This is the number every other model in this project has to beat
# to be a genuine improvement, not an assumed one.
from src.training.baseline import run_svm_baseline

baseline_summary, baseline_pooled = run_svm_baseline(
    df_m6, task="detection", embeddings=frozen_embeddings, max_folds=None)

print_header("Baseline (Frozen wav2vec 2.0 + Linear SVM) - Detection")
print_kv("Accuracy", f"{baseline_pooled['accuracy']:.4f}")
print_kv("F1", f"{baseline_pooled['f1']:.4f}")
print_kv("Recall (sensitivity)", f"{baseline_pooled['recall']:.4f}")
print_kv("Precision", f"{baseline_pooled['precision']:.4f}")
print_kv("Specificity", f"{baseline_pooled['specificity']:.4f}")
print_kv("AUROC", f"{baseline_pooled['auroc']:.4f}")

In [ ]:
# STAGE 4 - Phase 2, step 3: compare the baseline against the trained
# variants - Frozen wav2vec + MLP, LoRA wav2vec + MLP, MFCC CNN, and the full
# Fusion model - on the SAME task and fold protocol.
#
# NOTE ON SCALE: this cell runs at DEMONSTRATION scale (few folds, few
# epochs, AND a capped sample count per split) to prove the comparison
# pipeline is correct end to end for every variant - capping folds alone
# is not enough, since each fold would still run full-size epochs through
# wav2vec. This is not a real result. A full 28-fold LOSO run of
# deep_lora/fusion fine-tunes wav2vec 2.0 on the complete ~20k-utterance
# training split per fold and is hours of GPU time, not minutes - rerun
# this cell with max_folds=None, limit_samples=None, and epochs=20+ once
# you're ready to spend that time (Phase 3's real ablation table).
from src.training.runner import TrainingConfig, run_training

DEMO_MAX_FOLDS = 3
DEMO_EPOCHS = 5
DEMO_LIMIT_SAMPLES = 300

comparison_pooled = {"baseline_svm": baseline_pooled}

for model_name in ["deep_frozen", "deep_lora", "acoustic", "fusion"]:
    cfg = TrainingConfig(
        task="detection", model=model_name,
        epochs=DEMO_EPOCHS, max_folds=DEMO_MAX_FOLDS,
        limit_samples=DEMO_LIMIT_SAMPLES,
        run_name=f"detection_{model_name}",
    )
    _, pooled = run_training(df_m6, cfg)
    comparison_pooled[model_name] = pooled

In [ ]:
# STAGE 5 - Phase 2 comparison table: baseline SVM vs. every trained variant,
# pooled metrics side by side. Saved to outputs/metrics/phase2_comparison.csv
# for the paper/report; rerun after Stage 4 is re-run at full scale to get
# the final version of this table.
comparison_df = pd.DataFrame(comparison_pooled).T
comparison_df.index.name = "model"

comparison_path = config.METRICS_DIR / "phase2_comparison.csv"
comparison_df.to_csv(comparison_path)

print_header("Phase 2 Comparison - Detection")
print_kv("Saved to", comparison_path)
comparison_df